# Tutorial 02 · Build your design with SweetPea

In Session 1 every trial of your block was drawn at random. Now **you** design the block - and in Session 3 your group's design competes against the designs of the other groups.

**The target:** measure the **Stroop effect** and the **congruency sequence effect** (the Stroop effect after a congruent trial minus the Stroop effect after an incongruent trial) as well as possible with **96 trials**. How to judge "as well as possible" is the topic of Session 3 - today you build a first draft.

You will build the design step by step with [SweetPea](https://sweetpea-org.github.io/):
1. factors and a derived factor (congruency)
2. a transition (congruency of the previous trial)
3. a crossing and a budget (what must be balanced?)
4. a fully crossed design of the last four trials
5. a constraint (no repetition of the ink colour)
6. the first draft of your group design

All SweetPea code is yours to write - every exercise shows the pattern you need. Keep the documentation open:
- [Factorial experiment design](https://sweetpea-org.github.io/guide/factorial_design.html) - the ideas behind SweetPea (with the Stroop task as example)
- API reference: [factors and levels](https://sweetpea-org.github.io/api/factors.html) · [derivations](https://sweetpea-org.github.io/api/derivations.html) (`WithinTrial`, `Transition`, `Window`) · [constraints](https://sweetpea-org.github.io/api/constraints.html) · [blocks](https://sweetpea-org.github.io/api/main.html) (`CrossBlock`)

Sampling and inspecting sequences is ready-made (`synthesize`, `design_summary`, `count_sequences`).

## Setup

Run the next two cells once. The first one finds the course folder, so that Python can load the course tools from `resources/`.

In [ ]:
import os
import sys

# the course folder is one level above tutorials/ (two levels above tutorials/solutions/)
root = os.path.abspath("..")
if not os.path.isdir(os.path.join(root, "resources")):
    root = os.path.abspath(os.path.join("..", ".."))

sys.path.insert(0, root)

In [ ]:
from sweetpea import Factor, DerivedLevel
from sweetpea import WithinTrial, Transition, Window
from sweetpea import CrossBlock, MinimumTrials, Exclude, AtMostKInARow

from resources.design_tools import synthesize, design_summary, count_sequences, save_design

## 1. Factors and a derived factor

A **factor** is what we vary, its **levels** are the values it can take. Our Stroop task has two factors - the **word** and the **ink** colour - with the same four levels.

### Exercise 1.1
Define the factors `word` and `ink`, each with the levels in `colors`.

```python
Factor("name of the factor", [level_1, level_2, ...])
```

In [ ]:
colors = ["red", "green", "blue", "yellow"]

# ADD YOUR CODE HERE
word = ...
ink = ...

**Congruency** is not set directly - it follows from word and ink. In SweetPea that is a **derived factor**. Each of its levels needs a small function that answers *yes* (`True`) or *no* (`False`) for one trial. The two functions are given:

In [ ]:
def is_congruent(word, ink):
    return word == ink


def is_incongruent(word, ink):
    return word != ink

### Exercise 1.2
Define the derived factor `congruency` with the levels `"congruent"` and `"incongruent"`. A level that is decided **within one trial** uses `WithinTrial`:

```python
Factor("name of the factor", [
    DerivedLevel("name of level 1", WithinTrial(function_1, [factor_a, factor_b])),
    DerivedLevel("name of level 2", WithinTrial(function_2, [factor_a, factor_b])),
])
```

In [ ]:
# ADD YOUR CODE HERE
congruency = ...

### Exercise 1.3
A first design. A SweetPea block needs three lists:
- **design**: all factors of a trial (here: word, ink and congruency)
- **crossing**: the factors whose combinations must all appear equally often (here: word and ink - every combination once)
- **constraints**: extra rules (none yet: an empty list `[]`)

```python
CrossBlock(design, crossing, constraints)
```

In [ ]:
# ADD YOUR CODE HERE
design = ...
crossing = ...
constraints = ...

block = CrossBlock(design, crossing, constraints)

`synthesize` samples one sequence from the block, `design_summary` shows how balanced it is.

In [ ]:
sequence = synthesize(block)[0]

design_summary(sequence)

**Question:** How many of the 16 trials are congruent? Why? What would you change to get 50% congruent trials?

## 2. A transition: the previous trial

The congruency sequence effect compares the Stroop effect **after** a congruent and **after** an incongruent trial. That is a derived factor that looks at **two** trials - a `Transition`.

Its function gets the congruency of both trials as a short list: `congruency[0]` is the current trial, `congruency[-1]` the previous one.

In [ ]:
def previous_was_congruent(congruency):
    return congruency[-1] == "congruent"


def previous_was_incongruent(congruency):
    return congruency[-1] == "incongruent"


### Exercise 2.1
Define the factor `previous` with the levels `"after C"` and `"after I"`. It works like Exercise 1.2, with `Transition` instead of `WithinTrial` and only one factor in the list:

```python
DerivedLevel("name of the level", Transition(function, [factor]))
```

In [ ]:
# ADD YOUR CODE HERE
previous = ...

### Exercise 2.2
Build a block that crosses **congruency × previous** - all four cells of the 2 × 2 design from the lecture.
- design: all factors, now including `previous`
- crossing: `congruency` and `previous`
- constraints: fill the budget with `MinimumTrials(96)`

In [ ]:
# ADD YOUR CODE HERE
design = ...
crossing = ...
constraints = ...

block_2 = CrossBlock(design, crossing, constraints)

In [ ]:
sequence_2 = synthesize(block_2)[0]

design_summary(sequence_2)

## 3. What is not crossed is not balanced

Look at the ink colours and the repetitions in the summary above: SweetPea balances **only the crossing**. Everything else is whatever happens to satisfy the design.

### Exercise 3
1. Change the crossing so that the **ink colours** are balanced as well.
2. How many cells does the crossing have now, and how many repetitions of it fit into 96 trials?

In [ ]:
# ADD YOUR CODE HERE
design = ...
crossing = ...
constraints = ...

block_3 = CrossBlock(design, crossing, constraints)

In [ ]:
sequence_3 = synthesize(block_3)[0]

design_summary(sequence_3)

## 4. Fully crossed: the last four trials

In the lecture, every trial of a sequence was a factor with two levels (C, I). Fully crossing the current trial and the three before it gives 2 × 2 × 2 × 2 = 2⁴ = 16 sequences.

We already have the current trial (`congruency`) and the previous one (`previous`). For trials further back we need a `Window`: like a `Transition`, but it looks at more trials. In the function, `congruency[-2]` is the trial two back and `congruency[-3]` the trial three back.

In [ ]:
def congruent_two_back(congruency):
    return congruency[-2] == "congruent"


def incongruent_two_back(congruency):
    return congruency[-2] == "incongruent"


def congruent_three_back(congruency):
    return congruency[-3] == "congruent"


def incongruent_three_back(congruency):
    return congruency[-3] == "incongruent"


### Exercise 4.1
Define the factors `two_back` (levels `"C two back"`, `"I two back"`) and `three_back` (levels `"C three back"`, `"I three back"`). The `width` of a window is the number of trials it looks at, including the current one:

```python
DerivedLevel("name of the level", Window(function, [factor], width=3))
```

In [ ]:
# ADD YOUR CODE HERE
two_back = ...
three_back = ...

### Exercise 4.2
1. How many cells does the crossing **congruency × previous × two back × three back** have? How often does each cell fit into 96 trials?
2. Build the block.

In [ ]:
# ADD YOUR CODE HERE
design = ...
crossing = ...
constraints = ...

block_4 = CrossBlock(design, crossing, constraints)

`count_sequences` counts how often each sequence of four trials occurs (e.g. `CCCI` = three congruent trials, then an incongruent one). Sampling takes a few seconds.

In [ ]:
sequence_4 = synthesize(block_4)[0]

count_sequences(sequence_4)

**Questions:**
1. Some sequences occur fewer than 6 times. Why? (Hint: which trials have no three trials before them?)
2. With 6 trials per sequence, how precisely can a single player's response time for one sequence be estimated?
3. What would you need to cross the last **five** trials?

## 5. A constraint: no repetition of the ink colour

Repeating the ink colour means repeating the **response** - that alone makes responses faster and can fake a sequence effect (Mayr et al., 2003). Many studies therefore exclude such repetitions. (If repetitions are what you want to study, you would do the opposite - it depends on the research question.)

These functions get the ink of the previous trial (`ink[-1]`) and of the current trial (`ink[0]`):

In [ ]:
def ink_repeats(ink):
    return ink[-1] == ink[0]


def ink_switches(ink):
    return ink[-1] != ink[0]


### Exercise 5.1
Define the factor `ink_repetition` with the levels `"repeat"` and `"switch"` (a `Transition`, as in Exercise 2.1).

In [ ]:
# ADD YOUR CODE HERE
ink_repetition = ...

### Exercise 5.2
Take your design from Exercise 3 and **exclude** the ink repetitions:
- add `ink_repetition` to the design
- add the constraint that excludes one level of a factor:

```python
Exclude(factor["name of the level"])
```

Then check the summary: are there still ink repetitions? What happened to the word repetitions?

In [ ]:
# ADD YOUR CODE HERE
design = ...
crossing = ...
constraints = ...

block_5 = CrossBlock(design, crossing, constraints)

In [ ]:
sequence_5 = synthesize(block_5)[0]

design_summary(sequence_5)

## 6. A first draft of your group design

Now build the first draft of your group design. The target: measure the **Stroop effect** and the **sequence effect** as well as possible with 96 trials. Ideas:
- which crossing balances what both effects need?
- which confounds should be excluded (ink repetitions, ...)?
- at most 3 congruent or incongruent trials in a row: `AtMostKInARow(3, congruency)`
- more constraints in the [constraints documentation](https://sweetpea-org.github.io/api/constraints.html)

Inspect every sequence with `design_summary`.

**Rules:** 96 trials, colours red / green / blue / yellow. In Session 3 you learn how to judge designs **before** any data is collected - and what that judgement misses.

In [ ]:
# ADD YOUR CODE HERE
design = ...
crossing = ...
constraints = ...

block_group = CrossBlock(design, crossing, constraints)

In [ ]:
my_design = synthesize(block_group)[0]

design_summary(my_design)

Save the design under your group name. It is written to `designs/<group name>.csv`.

In [ ]:
GROUP = "group1"   # ADD YOUR GROUP NAME HERE

path = os.path.join(root, "designs", GROUP + ".csv")
save_design(my_design, path)

Try your design once yourself as a practice block (in a terminal, in the course folder; nothing is saved):

```
python -m experiments.stroop --practice --design designs/<group name>.csv
```

## Take-away

- The **crossing** decides what is balanced; the congruency sequence effect needs all four cells of current × previous trial.
- **Fully crossing** longer histories is possible, but every extra trial doubles the cells - and halves the trials per cell.
- SweetPea balances only what you **ask** for - always inspect the sequence you got.
- Constraints remove confounds (repetitions of the ink colour) but also remove sequences.
- Is your design better than a random block? Session 3: measuring design quality before and after data collection.